<a href="https://colab.research.google.com/github/3OMDEH/Machine-Learning-Projects-Beginner-/blob/main/TwitterSentimentAnalaysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kaggle**

In [1]:
!pip install kaggle

In [2]:
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")

Using Colab cache for faster access to the 'sentiment140' dataset.


# **Importing Libraries**

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re # provides regular expression operations

from nltk.corpus import stopwords
"""imports a list of common 'stop words' (e.g., 'the', 'is', 'and') from the
Natural Language Toolkit (NLTK) library,which are often removed from text data before analysis."""

from nltk.stem.porter import PorterStemmer
"""
used to reduce words to their base or root form (e.g., 'running' to 'run').
"""

from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF is a common way to represent text data numerically for machine learning models.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
import nltk
nltk.download('stopwords') # Downloading the Stopwords Corpus
# Stopwords are often removed from the text before analysis because they typically don't carry significant meaning.

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# **Reading & Exploring the Data**

In [6]:
#@title Reading the path
file_list = os.listdir(path)
print(file_list)

['training.1600000.processed.noemoticon.csv']


In [7]:
#@title Converting to pandas df
col_names = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv(os.path.join(path, file_list[0]), names=col_names, encoding='ISO-8859-1') # constructs the full path to a CSV file by joining the path and the first element of file_list.
df.head(5)

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [8]:
df.info()

"""
No missing values were found
No Duplicates were found
"""

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   id      1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


In [9]:
#@title Replacing 4 with 1
"""
0 --> Negative
4 --> Positive

The conversion makes the data more compatible with standard
classification algorithms that expect binary labels (0 or 1).
"""

print(df['target'].value_counts()) # Before
df.replace({'target':{4:1}}, inplace=True)
print(df['target'].value_counts()) # After
#

target
0    800000
4    800000
Name: count, dtype: int64
target
0    800000
1    800000
Name: count, dtype: int64


# **Stemming: Pre-processing step**

Is a process of reducing the word to its root word. For example, 'running', 'runs', and 'runner' would all be reduced to 'run'.

In [10]:
port_stem = PorterStemmer() # PorterStemmer is a widely used algorithm in NLP for 'stemming' words.

In [11]:
def stemming(content):
  """
  This function will be applied to each sample individually
  """
  stemmed_content = re.sub('[^a-zA-Z]', ' ', content) #  remove all characters that are not English alphabet letters (symbols, punctuations, marks)
  stemmed_content = stemmed_content.lower() # Ensures that each letter treated the same to avoid case-sensitivity
  stemmed_content = stemmed_content.split() #  A basic form of tokenization, breaks the cleaned string into a list of words

  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  """
  Stemming is applied to all words EXCEPT stopwords since they do not carry
  meaningful information for sentiment analysis.
  """
  stemmed_content = ' '.join(stemmed_content) # joins the processed list of stemmed words back into a single string, with each word separated by a space
  return stemmed_content


In [12]:
df['stemmed_text'] = df['text'].apply(stemming)

In [13]:
df['stemmed_text'].head(10)

,stemmed_text
0,switchfoot http twitpic com zl awww bummer sho...
1,upset updat facebook text might cri result sch...
2,kenichan dive mani time ball manag save rest g...
3,whole bodi feel itchi like fire
4,nationwideclass behav mad see
5,kwesidei whole crew
6,need hug
7,loltrish hey long time see ye rain bit bit lol...
8,tatiana k nope
9,twittera que muera


# **Separating the Data**

In [31]:
x = df['stemmed_text']
y = df['target']

In [32]:
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    stratify=y,
                                                    random_state=2)

In [33]:
#@title Creating Embeddings using TF-IDF

vectorizer = TfidfVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)



"""
When you fit_transform on the test set, the vectorizer re-learns the
vocabulary and IDF values from the test set, which can lead to data leakage
and an inaccurate evaluation of your model.


Term Frequency (TF) measures how frequently a term appears in a document,
while Inverse Document Frequency (IDF) measures how important
a term is across the entire corpus.

Words that are common across many documents (like 'the', 'a') get a lower IDF
score, while unique or specific words get a higher score.
"""

"\nWhen you fit_transform on the test set, the vectorizer re-learns the \nvocabulary and IDF values from the test set, which can lead to data leakage \nand an inaccurate evaluation of your model.\n\n\nTerm Frequency (TF) measures how frequently a term appears in a document,\nwhile Inverse Document Frequency (IDF) measures how important \na term is across the entire corpus. \n\nWords that are common across many documents (like 'the', 'a') get a lower IDF \nscore, while unique or specific words get a higher score.\n"

In [34]:
print(x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9453092 stored elements and shape (1280000, 461488)>
  Coords	Values
  (0, 436713)	0.27259876264838384
  (0, 354543)	0.3588091611460021
  (0, 185193)	0.5277679060576009
  (0, 109306)	0.3753708587402299
  (0, 235045)	0.41996827700291095
  (0, 443066)	0.4484755317023172
  (1, 160636)	1.0
  (2, 109306)	0.4591176413728317
  (2, 124484)	0.1892155960801415
  (2, 407301)	0.18709338684973031
  (2, 129411)	0.29074192727957143
  (2, 406399)	0.32105459490875526
  (2, 433560)	0.3296595898028565
  (2, 77929)	0.31284080750346344
  (2, 443430)	0.3348599670252845
  (2, 266729)	0.24123230668976975
  (2, 409143)	0.15169282335109835
  (2, 178061)	0.1619010109445149
  (2, 150715)	0.18803850583207948
  (2, 132311)	0.2028971570399794
  (2, 288470)	0.16786949597862733
  (3, 406399)	0.29029991238662284
  (3, 158711)	0.4456939372299574
  (3, 151770)	0.278559647704793
  (3, 56476)	0.5200465453608686
  :	:
  (1279996, 318303)	0.21254698865277744
  (12

# **Creating & Training the Model**

In [68]:
Logistic_model = LogisticRegression(max_iter=1000)
Logistic_model.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

In [69]:

train_accuracy = Logistic_model.score(x_train, y_train)
test_accuracy = Logistic_model.score(x_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

"""
The .score() method for scikit-learn classifiers typically returns the mean accuracy on the given test data and labels.
"""

# The Test Accuracy is already calculated in the next cell.

Training Accuracy: 0.7987
Test Accuracy: 0.7767


In [70]:
 from sklearn.metrics import log_loss

# For computing loss: Returns the probability estimates for each class for every sample in x_test
y_pred_proba = Logistic_model.predict_proba(x_test)[:, 1] # [:, 1] returns the probabilitis for the positive class

# For computing accuracy: Stores the hard class predictions (0 or 1), [Not the probabilities]
y_pred_int = Logistic_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred_int)
print(f"Test Accuracy: {accuracy:.4f}")

loss = log_loss(y_test, y_pred)
print(f"Log Loss: {loss:.4f}")

"""
A log loss of {0.47} falls between perfect and random, indicating that the
model is performing better than random chance.
"""

Accuracy: 0.7767
Log Loss: 0.4737


'\nA log loss of {0.47} falls between perfect and random, indicating that the \nmodel is performing better than random chance.\n'

In [71]:
print(y_pred_proba)

[0.89117899 0.8688888  0.47197962 ... 0.96757807 0.15314119 0.94324036]


In [72]:
print(y_pred_int)

[1 1 0 ... 1 0 1]


# **Saving the Trained Model**

In [73]:
import pickle

In [74]:
filename = "trained_model.sav" # Defines the file name the model will saved in.
pickle.dump(Logistic_model, open(filename, 'wb'))


"""
Why is this important? Saving your trained model allows you to:

Reuse it later: You don't have to retrain the model every time you want to use it for predictions on new data.
Deploy it: You can load the saved model into another application or environment to make predictions.
Share it: You can easily share your trained model with others without needing to share your entire training pipeline.

"""

"\nWhy is this important? Saving your trained model allows you to:\n\nReuse it later: You don't have to retrain the model every time you want to use it for predictions on new data.\nDeploy it: You can load the saved model into another application or environment to make predictions.\nShare it: You can easily share your trained model with others without needing to share your entire training pipeline.\n\n"

# **Using the Saved Model for Future Predictions**

In [75]:
loaded_model = pickle.load(open('/content/trained_model.sav', 'rb'))

In [87]:
x_new = x_test[24]

In [88]:
print(y_test.iloc[24])

0


In [89]:
prediction = loaded_model.predict(x_new)

In [90]:
print(prediction)

[1]


In [91]:
if prediction[0] == 0:
  print("Negative Tweet")
else:
  print("Positive Tweet")

Positive Tweet
